# MAI511-2 – Advanced Machine Learning
## Lab Exercise 8: Demonstration of Hierarchical Clustering

**Name:** Sanjay R A  
**Register number (last 3 digits):** 555  
**Program:** Lab 8  

### Aim
To implement and analyse Hierarchical Clustering using Agglomerative and Divisive approaches and understand the impact of different distance measures and linkage methods on cluster formation.

### Dataset selected
The **Wine dataset** from Scikit-learn is used. It contains numerical chemical measurements of wine samples. This is suitable for clustering because the task is to group samples based on similarity without using the known wine class as a clustering input.

### Learning roadmap
1. Load and understand the dataset.
2. Clean and standardize the numerical features.
3. Calculate Euclidean, Manhattan and Cosine distance matrices.
4. Perform Agglomerative clustering with four linkage methods.
5. Draw dendrograms.
6. Perform Divisive clustering using Bisecting K-Means.
7. Try different cluster counts and compare results.
8. Visualize clusters using PCA.
9. Compare and interpret the results.

**Note:** The dataset's target class is retained only for reference; it is not used to form clusters.

## 1. Import the required libraries

Libraries are tools that provide ready-made functions. We use:
- `pandas` to handle tabular data.
- `numpy` for numerical operations.
- `matplotlib` and `seaborn` for graphs.
- `scikit-learn` for dataset loading, preprocessing, distances, clustering and PCA.
- `scipy` for dendrogram generation.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_wine
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import pairwise_distances, silhouette_score
from sklearn.cluster import AgglomerativeClustering, BisectingKMeans
from sklearn.decomposition import PCA

from scipy.cluster.hierarchy import linkage, dendrogram

print("All libraries imported successfully!")

## 2. Load the dataset and inspect it

`load_wine()` loads a built-in dataset. We convert its features into a Pandas DataFrame so that rows and columns are easy to inspect.

- **Records:** number of wine samples (rows).
- **Attributes:** number of measured features (columns).
- **Data types:** type of each column.

In [ ]:
wine = load_wine()

# Create a table containing only the numerical input features
df = pd.DataFrame(wine.data, columns=wine.feature_names)

# Keep the known class separately for reference only
actual_class = wine.target

print("First five records:")
display(df.head())

print("Number of records:", df.shape[0])
print("Number of attributes:", df.shape[1])

print("\nData types:")
print(df.dtypes)

print("\nDataset information:")
df.info()

## 3. Preprocess the data

Preprocessing prepares data for clustering.

1. **Missing values:** replace missing numerical values with the column mean.
2. **Irrelevant attributes:** the dataset contains only useful numerical measurements. Sample names/IDs and the target class are not included as clustering features.
3. **Standardization:** `StandardScaler` transforms each feature to approximately mean 0 and standard deviation 1. This prevents large-scale features from dominating distance calculations.

In [ ]:
# Step 1: Handle missing values using the mean of each column
imputer = SimpleImputer(strategy="mean")
data_filled = imputer.fit_transform(df)

# Step 2: Standardize all numerical features
scaler = StandardScaler()
X = scaler.fit_transform(data_filled)

print("Shape after preprocessing:", X.shape)
print("Missing values after imputation:", np.isnan(X).sum())
print("First five standardized rows:")
display(pd.DataFrame(X, columns=df.columns).head())

## 4. Calculate pairwise distance matrices

A distance matrix shows how far each pair of records is from one another. Smaller distance means greater similarity.

- **Euclidean distance:** straight-line distance.
- **Manhattan distance:** sum of absolute differences.
- **Cosine distance:** based on the angle between two vectors; it is `1 - cosine similarity`.

For readability, we display only the first 5 × 5 section of each matrix. The full matrices are still calculated.

In [ ]:
distance_methods = ["euclidean", "manhattan", "cosine"]
distance_matrices = {}

for method in distance_methods:
    matrix = pairwise_distances(X, metric=method)
    distance_matrices[method] = matrix

    print("\n", method.upper(), "distance (first 5 records):")
    display(pd.DataFrame(matrix[:5, :5]).round(3))

## 5. Agglomerative Hierarchical Clustering

Agglomerative clustering is a **bottom-up** method:
1. Start with every data point as an individual cluster.
2. Find the two closest clusters.
3. Merge them.
4. Repeat until the required number of clusters remains.

### Linkage methods
- **Single:** minimum distance between points in two clusters.
- **Complete:** maximum distance between points in two clusters.
- **Average:** average distance between all pairs of points in two clusters.
- **Ward:** merges clusters that cause the smallest increase in within-cluster variance. Ward linkage uses Euclidean distance.

In [ ]:
# Use 3 clusters for the first comparison
number_of_clusters = 3

linkage_methods = ["single", "complete", "average", "ward"]
agglomerative_results = {}

for method in linkage_methods:
    model = AgglomerativeClustering(
        n_clusters=number_of_clusters,
        linkage=method
    )
    labels = model.fit_predict(X)
    agglomerative_results[method] = labels

    score = silhouette_score(X, labels)
    print(method.title(), "linkage")
    print("Cluster sizes:", pd.Series(labels).value_counts().sort_index().to_dict())
    print("Silhouette score:", round(score, 3))
    print("-" * 40)

## 6. Draw dendrograms

A **dendrogram** is a tree-shaped diagram that shows the order in which clusters are merged. The vertical height of a merge represents the distance at which the merge occurs.

We use a small sample of 60 records to keep the diagrams clear and fast. The clustering models above still use the complete dataset.

In [ ]:
# Select the first 60 records for clear dendrograms
sample_data = X[:60]

plt.figure(figsize=(16, 5))
for position, method in enumerate(linkage_methods, 1):
    plt.subplot(2, 2, position)

    # Ward linkage is compatible with Euclidean distance
    tree = linkage(sample_data, method=method, metric="euclidean")
    dendrogram(tree, no_labels=True)

    plt.title(method.title() + " Linkage")
    plt.xlabel("Data points")
    plt.ylabel("Distance")

plt.tight_layout()
plt.show()

## 7. Divisive Hierarchical Clustering

Divisive clustering is a **top-down** approach:
1. Begin with all records in one cluster.
2. Split a cluster into smaller clusters.
3. Continue splitting until the required number of clusters is reached.

For this lab, we use Scikit-learn's `BisectingKMeans`. It repeatedly divides a cluster into two groups, making it a practical divisive clustering approach. It is not the same as every possible divisive hierarchical algorithm, but it demonstrates the top-down splitting idea.

In [ ]:
divisive_model = BisectingKMeans(
    n_clusters=3,
    random_state=42,
    n_init=10
)

divisive_labels = divisive_model.fit_predict(X)

print("Divisive clustering (Bisecting K-Means)")
print("Cluster sizes:")
print(pd.Series(divisive_labels).value_counts().sort_index())
print("Silhouette score:", round(silhouette_score(X, divisive_labels), 3))

## 8. Experiment with different numbers of clusters

The number of clusters affects how the data is grouped. We test 2, 3, 4 and 5 clusters.

The **silhouette score** ranges from -1 to +1:
- Near +1: clusters are well separated.
- Around 0: clusters overlap.
- Below 0: some records may be assigned to an unsuitable cluster.

A higher score indicates better separation according to this measure; it does not automatically mean the clusters are meaningful in the real world.

In [ ]:
cluster_counts = [2, 3, 4, 5]
experiment_rows = []

for k in cluster_counts:
    # Agglomerative clustering with Ward linkage
    agg = AgglomerativeClustering(n_clusters=k, linkage="ward")
    agg_labels = agg.fit_predict(X)

    # Divisive clustering
    div = BisectingKMeans(n_clusters=k, random_state=42, n_init=10)
    div_labels = div.fit_predict(X)

    experiment_rows.append({
        "Number of clusters": k,
        "Agglomerative silhouette": silhouette_score(X, agg_labels),
        "Divisive silhouette": silhouette_score(X, div_labels)
    })

experiment_table = pd.DataFrame(experiment_rows)
display(experiment_table.round(3))

## 9. Visualize final clusters using PCA

The original dataset has 13 features, so it cannot be directly shown on a simple 2D graph.

**PCA (Principal Component Analysis)** reduces the 13 features to 2 new components while retaining as much variation as possible. We use these two components only for visualization.

The plotted colours represent cluster assignments, not the original wine classes.

In [ ]:
# Reduce the 13 standardized features to 2 dimensions
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X)

# Fit the final Agglomerative and Divisive models
agg_final = AgglomerativeClustering(n_clusters=3, linkage="ward")
agg_final_labels = agg_final.fit_predict(X)

div_final = BisectingKMeans(n_clusters=3, random_state=42, n_init=10)
div_final_labels = div_final.fit_predict(X)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(X_pca[:, 0], X_pca[:, 1], c=agg_final_labels, cmap="viridis")
axes[0].set_title("Agglomerative Clustering (Ward)")
axes[0].set_xlabel("Principal Component 1")
axes[0].set_ylabel("Principal Component 2")

axes[1].scatter(X_pca[:, 0], X_pca[:, 1], c=div_final_labels, cmap="viridis")
axes[1].set_title("Divisive Clustering (Bisecting K-Means)")
axes[1].set_xlabel("Principal Component 1")
axes[1].set_ylabel("Principal Component 2")

plt.tight_layout()
plt.show()

print("Variance explained by the two PCA components:",
      round(pca.explained_variance_ratio_.sum() * 100, 2), "%")

## 10. Compare distance measures and linkage methods

To compare distance measures fairly, we use the same number of clusters (3) and the same linkage (average), changing only the distance metric. Then we compare the four linkage methods using Euclidean distance.

**Important:** Ward linkage is restricted to Euclidean distance in this experiment.

In [ ]:
comparison_rows = []

# Compare distance measures while keeping average linkage fixed
for metric in ["euclidean", "manhattan", "cosine"]:
    model = AgglomerativeClustering(
        n_clusters=3,
        linkage="average",
        metric=metric
    )
    labels = model.fit_predict(X)
    comparison_rows.append({
        "Experiment": "Distance: " + metric,
        "Silhouette score": silhouette_score(X, labels)
    })

# Compare linkage methods while keeping Euclidean distance fixed
for method in ["single", "complete", "average", "ward"]:
    model = AgglomerativeClustering(
        n_clusters=3,
        linkage=method
    )
    labels = model.fit_predict(X)
    comparison_rows.append({
        "Experiment": "Linkage: " + method,
        "Silhouette score": silhouette_score(X, labels)
    })

comparison_table = pd.DataFrame(comparison_rows)
display(comparison_table.round(3))

plt.figure(figsize=(11, 5))
plt.bar(comparison_table["Experiment"], comparison_table["Silhouette score"])
plt.xticks(rotation=35, ha="right")
plt.ylabel("Silhouette score")
plt.title("Comparison of Distance Measures and Linkage Methods")
plt.tight_layout()
plt.show()

## 11. Interpret the clusters and compare the approaches

Use the cluster sizes and silhouette scores printed above to describe the output from your own notebook run.

### Agglomerative vs Divisive

| Agglomerative | Divisive |
|---|---|
| Bottom-up approach | Top-down approach |
| Starts with each record as a separate cluster | Starts with all records in one cluster |
| Repeatedly merges the closest clusters | Repeatedly splits clusters |
| Dendrogram naturally shows the merging process | Bisecting K-Means demonstrates repeated splitting |
| Different linkage methods can change the result | Splitting decisions can change the result |

### Interpretation guide
- A cluster groups wine samples with similar standardized chemical measurements.
- Different distance measures define similarity differently, so assignments can change.
- Single linkage may create long, chain-like clusters.
- Complete linkage tends to favour compact clusters.
- Average linkage uses the average pairwise distance.
- Ward linkage tends to form compact groups by minimizing increases in within-cluster variance.
- Compare the silhouette scores rather than assuming one method is always best.

**Conclusion:** Hierarchical clustering reveals nested groupings of similar records. Agglomerative clustering builds groups by merging, while the divisive Bisecting K-Means approach builds groups by splitting. The choice of distance measure, linkage method and number of clusters affects the final assignments.

## 12. Self-learning questions with answers

### Q1. What is clustering?
**Answer:** Clustering is an unsupervised machine learning technique that groups similar data points together. It does not require pre-labelled output classes.

### Q2. What is hierarchical clustering?
**Answer:** It is a clustering method that creates a hierarchy of nested clusters. The hierarchy can be represented using a dendrogram.

### Q3. What are the two main types of hierarchical clustering?
**Answer:** Agglomerative (bottom-up merging) and Divisive (top-down splitting).

### Q4. What is the difference between supervised and unsupervised learning?
**Answer:** Supervised learning uses labelled examples to predict an output. Unsupervised learning works with data that has no target labels and discovers patterns such as clusters.

### Q5. Why is data standardization required?
**Answer:** Features may have different units and ranges. Standardization places them on a comparable scale so that large-valued features do not dominate distance calculations.

### Q6. What is Euclidean distance?
**Answer:** It is the straight-line distance between two points. For two dimensions, it is √((x₂ − x₁)² + (y₂ − y₁)²).

### Q7. What is Manhattan distance?
**Answer:** It is the sum of the absolute differences between corresponding feature values.

### Q8. What is Cosine distance?
**Answer:** It is 1 minus cosine similarity. It measures the difference in direction between two vectors.

### Q9. What is linkage?
**Answer:** Linkage defines how the distance between two clusters is calculated. Common methods are single, complete, average and Ward.

### Q10. Explain single, complete and average linkage.
**Answer:** Single uses the minimum pairwise distance, complete uses the maximum pairwise distance, and average uses the average pairwise distance between two clusters.

### Q11. What is Ward linkage?
**Answer:** Ward linkage merges the pair of clusters that produces the smallest increase in within-cluster variance. It is used with Euclidean distance.

### Q12. What is a dendrogram?
**Answer:** A dendrogram is a tree diagram that displays the order and distance of cluster merges. Cutting the tree at a selected height gives a chosen number of clusters.

### Q13. What is divisive clustering?
**Answer:** Divisive clustering starts with all records in one cluster and repeatedly divides clusters into smaller groups.

### Q14. What is Bisecting K-Means?
**Answer:** It is a divisive-style clustering algorithm that repeatedly splits a selected cluster into two using K-Means until the requested number of clusters is obtained.

### Q15. What is a silhouette score?
**Answer:** It measures how similar a point is to its own cluster compared with other clusters. Its range is -1 to +1; higher values generally indicate better-separated clusters.

### Q16. Why is PCA used in this experiment?
**Answer:** PCA reduces high-dimensional data to two components so that clusters can be displayed on a 2D scatter plot.

### Q17. Does PCA perform clustering?
**Answer:** No. PCA is a dimensionality-reduction method. It transforms features for analysis or visualization; the clustering algorithm assigns the cluster labels.

### Q18. Why is the wine target class not used?
**Answer:** The task is unsupervised clustering. Using the target class as an input would leak known labels into the clustering process.

### Q19. Does a higher silhouette score always mean the clusters are useful?
**Answer:** No. It indicates geometric separation under the chosen distance measure, but domain knowledge is still needed to decide whether the groups are meaningful.

### Q20. What happens when the number of clusters changes?
**Answer:** The algorithm divides or merges the data into a different number of groups. Cluster membership and cluster sizes can therefore change.

## 13. Viva preparation – quick revision

| Term | One-line answer |
|---|---|
| Dataset | A collection of data records |
| Feature | An input attribute or column |
| Preprocessing | Preparing data before analysis |
| StandardScaler | Converts features to mean 0 and standard deviation 1 |
| Distance matrix | Table of pairwise distances |
| Agglomerative | Bottom-up clustering |
| Divisive | Top-down clustering |
| Linkage | Rule for measuring distance between clusters |
| Dendrogram | Tree diagram of hierarchical merging |
| PCA | Reduces the number of dimensions |
| Silhouette score | Measures cluster separation and cohesion |

### Important exam explanation
If asked to explain the program, describe the flow in this order:

**Load dataset → inspect records and attributes → handle missing values → standardize → calculate distances → perform Agglomerative clustering → draw dendrograms → perform Divisive clustering → test cluster counts → visualize with PCA → compare scores → interpret clusters.**

## 14. Optional Streamlit visualization (for the rubric)

This cell creates `app.py`. The app uses Plotly charts with a dark theme, a PCA scatter plot, cluster-size chart, and silhouette distribution.

Install dependencies and run from the same folder:

```bash
pip install streamlit scikit-learn pandas plotly
streamlit run app.py
```


In [ ]:
streamlit_code = '''
import streamlit as st
import pandas as pd
import plotly.express as px
from sklearn.datasets import load_wine
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import AgglomerativeClustering, BisectingKMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, silhouette_samples

st.set_page_config(
    page_title="Hierarchical Clustering Explorer",
    page_icon="🌿",
    layout="wide",
)

# Use a dark Plotly theme so charts match Streamlit's dark interface.
st.title("Hierarchical Clustering Explorer")
st.caption("Explore Agglomerative and Divisive-style clustering on the Wine dataset.")

wine = load_wine(as_frame=True)
df = wine.data
X = StandardScaler().fit_transform(df)

with st.sidebar:
    st.header("Clustering settings")
    method = st.selectbox(
        "Choose clustering approach",
        ["Agglomerative", "Divisive (Bisecting K-Means)"],
    )
    k = st.slider("Number of clusters", min_value=2, max_value=6, value=3)

if method == "Agglomerative":
    model = AgglomerativeClustering(n_clusters=k, linkage="ward")
else:
    model = BisectingKMeans(n_clusters=k, random_state=42, n_init=10)

labels = model.fit_predict(X)
score = silhouette_score(X, labels)
sizes = pd.Series(labels, name="Cluster").value_counts().sort_index()
sil_values = silhouette_samples(X, labels)

pca = PCA(n_components=2, random_state=42)
points = pca.fit_transform(X)
plot_df = pd.DataFrame({
    "PC1": points[:, 0],
    "PC2": points[:, 1],
    "Cluster": labels.astype(str),
    "Wine class": wine.target.astype(str),
    "Silhouette": sil_values,
})

m1, m2, m3 = st.columns(3)
m1.metric("Silhouette score", f"{score:.3f}")
m2.metric("Clusters", k)
m3.metric("Samples", len(labels))

st.subheader("Cluster sizes")
size_df = sizes.rename("Count").reset_index()
size_df.columns = ["Cluster", "Count"]
fig_sizes = px.bar(
    size_df, x="Cluster", y="Count", color="Cluster",
    text="Count", template="plotly_dark",
    title="Number of wine samples in each cluster",
)
fig_sizes.update_layout(height=360, margin=dict(l=10, r=10, t=55, b=10))
fig_sizes.update_traces(textposition="outside")
st.plotly_chart(fig_sizes, use_container_width=True)

st.subheader("PCA cluster visualization")
fig_pca = px.scatter(
    plot_df, x="PC1", y="PC2", color="Cluster",
    hover_data=["Wine class", "Silhouette"],
    template="plotly_dark",
    title="Wine samples projected onto the first two principal components",
)
fig_pca.update_traces(marker=dict(size=9, opacity=0.85))
fig_pca.update_layout(height=560, margin=dict(l=10, r=10, t=55, b=10))
st.plotly_chart(fig_pca, use_container_width=True)

st.subheader("Silhouette values by cluster")
fig_sil = px.box(
    plot_df, x="Cluster", y="Silhouette", color="Cluster",
    points="outliers", template="plotly_dark",
    title="Within-cluster cohesion and separation",
)
fig_sil.add_hline(y=0, line_dash="dash", line_color="#f87171")
fig_sil.update_layout(height=420, margin=dict(l=10, r=10, t=55, b=10))
st.plotly_chart(fig_sil, use_container_width=True)

with st.expander("View cluster-size table"):
    st.dataframe(size_df.set_index("Cluster"), use_container_width=True)

st.caption(
    "Note: PCA is used only to display the clusters in 2D. "
    "Clustering and silhouette score are calculated using all standardized features."
)

'''

with open("app.py", "w", encoding="utf-8") as file:
    file.write(streamlit_code)

print("Streamlit application saved as app.py")


## 15. Result

The Wine dataset was loaded, inspected and preprocessed. Euclidean, Manhattan and Cosine pairwise distances were calculated. Agglomerative clustering was implemented with Single, Complete, Average and Ward linkage. Dendrograms were generated, and a divisive-style approach was demonstrated using Bisecting K-Means. Different cluster counts and methods were compared using silhouette scores, and PCA scatter plots were used to visualize the final clusters.

**Before submission:** Run all notebook cells from top to bottom, check that all outputs and graphs appear, and save the notebook using the required filename format.

# Additional Visualizations – Extended Analysis

Run these cells after the main notebook has completed. They add more graphs for analysis and viva demonstration.

## 16. Feature distribution histograms
Histograms show how numerical feature values are distributed.

In [ ]:
df.iloc[:, :6].hist(figsize=(14, 8), bins=15, edgecolor="black")
plt.suptitle("Distribution of First 6 Wine Features")
plt.tight_layout()
plt.show()

## 17. Correlation heatmap
Shows positive and negative relationships between numerical attributes.

In [ ]:
plt.figure(figsize=(13, 9))
sns.heatmap(df.corr(), cmap="coolwarm", center=0, linewidths=0.3)
plt.title("Correlation Heatmap of Wine Features")
plt.tight_layout()
plt.show()

## 18. Boxplots
Boxplots show median, spread and possible outliers. Standardized features are used for fair comparison.

In [ ]:
scaled_df = pd.DataFrame(X, columns=df.columns)
plt.figure(figsize=(15, 6))
sns.boxplot(data=scaled_df.iloc[:, :8])
plt.title("Boxplots of First 8 Standardized Features")
plt.xticks(rotation=35, ha="right")
plt.ylabel("Standardized value")
plt.tight_layout()
plt.show()

## 19. Pairwise distance heatmaps
These display the distances among the first 25 records. The diagonal is zero because each record has zero distance from itself.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
for ax, metric in zip(axes, ["euclidean", "manhattan", "cosine"]):
    sns.heatmap(distance_matrices[metric][:25, :25], ax=ax, cmap="viridis")
    ax.set_title(metric.title() + " Distance")
    ax.set_xlabel("Sample")
    ax.set_ylabel("Sample")
plt.suptitle("Distance Matrix Heatmaps – First 25 Samples")
plt.tight_layout()
plt.show()

## 20. Cluster size comparison
Compares how many records are assigned to each cluster.

In [ ]:
agg_sizes = pd.Series(agg_final_labels).value_counts().sort_index()
div_sizes = pd.Series(div_final_labels).value_counts().sort_index()
size_table = pd.DataFrame({"Agglomerative": agg_sizes, "Divisive": div_sizes}).fillna(0)
display(size_table)
size_table.plot(kind="bar", figsize=(9, 5))
plt.title("Cluster Size Comparison")
plt.xlabel("Cluster label")
plt.ylabel("Number of records")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 21. Silhouette score comparison
Higher silhouette values generally indicate better-separated clusters.

In [ ]:
plt.figure(figsize=(11, 5))
sns.barplot(data=comparison_table, x="Experiment", y="Silhouette score")
plt.axhline(0, linewidth=1)
plt.title("Silhouette Scores Across Experiments")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()

## 22. PCA explained variance
Shows how much of the dataset's variation is represented by each principal component.

In [ ]:
pca_all = PCA()
pca_all.fit(X)
variance = pca_all.explained_variance_ratio_
cumulative = np.cumsum(variance)
plt.figure(figsize=(9, 5))
plt.bar(range(1, len(variance) + 1), variance, label="Individual variance")
plt.plot(range(1, len(cumulative) + 1), cumulative, marker="o",
         label="Cumulative variance")
plt.xlabel("Principal component")
plt.ylabel("Explained variance ratio")
plt.title("PCA Explained Variance")
plt.legend()
plt.tight_layout()
plt.show()

## 23. Cluster feature profile heatmap
Shows the average standardized feature values in each Agglomerative cluster.

In [ ]:
cluster_profile = scaled_df.copy()
cluster_profile["Cluster"] = agg_final_labels
profile_means = cluster_profile.groupby("Cluster").mean()
plt.figure(figsize=(14, 5))
sns.heatmap(profile_means, cmap="coolwarm", center=0, annot=False)
plt.title("Average Standardized Feature Values by Cluster")
plt.xlabel("Feature")
plt.ylabel("Cluster")
plt.tight_layout()
plt.show()
display(profile_means.round(2))

## 24. Three-dimensional PCA visualization
Displays the Agglomerative clusters using three principal components.

In [ ]:
pca_3d = PCA(n_components=3)
X_pca_3d = pca_3d.fit_transform(X)
fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection="3d")
points = ax.scatter(X_pca_3d[:, 0], X_pca_3d[:, 1], X_pca_3d[:, 2],
                    c=agg_final_labels, cmap="viridis", s=45)
ax.set_title("3D PCA – Agglomerative Clusters")
ax.set_xlabel("Principal Component 1")
ax.set_ylabel("Principal Component 2")
ax.set_zlabel("Principal Component 3")
fig.colorbar(points, ax=ax, label="Cluster")
plt.tight_layout()
plt.show()
print("Variance retained by 3 components:",
      round(pca_3d.explained_variance_ratio_.sum() * 100, 2), "%")

## 25. Dendrogram with a cut line
The horizontal line illustrates one possible cut through the hierarchy. Its height is illustrative, not a guaranteed cluster count.

In [ ]:
ward_tree = linkage(X[:60], method="ward", metric="euclidean")
plt.figure(figsize=(15, 6))
dendrogram(ward_tree, no_labels=True)
plt.axhline(y=15, linestyle="--", linewidth=2, label="Illustrative cut height")
plt.title("Ward Dendrogram with Cluster Cut Line")
plt.xlabel("Data points")
plt.ylabel("Merge distance")
plt.legend()
plt.tight_layout()
plt.show()

## Visualization viva guide

- **Histogram:** shows how feature values are distributed.
- **Correlation heatmap:** shows relationships between features.
- **Boxplot:** shows median, spread and possible outliers.
- **Distance heatmap:** shows pairwise distances between records.
- **Cluster-size chart:** compares the number of records in each group.
- **Silhouette chart:** compares cluster separation scores.
- **PCA variance chart:** shows how much information PCA retains.
- **Cluster-profile heatmap:** helps explain how clusters differ.
- **3D PCA plot:** shows clusters in three reduced dimensions.
- **Dendrogram cut line:** illustrates how a hierarchy can be separated.

**Viva tip:** For every graph, explain its axes, what the colours represent, and one observation from the output.